In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
arqlmed = spark.read.table("hive_metastore.silver.silver_arqlmed")
medidas = spark.read.table("hive_metastore.silver.silver_medidas")
mds     = spark.read.table("hive_metastore.bronze.bronze_mds")
weather = spark.read.table("hive_metastore.silver.silver_weather")
event_log = spark.read.table("hive_metastore.silver.silver_event_log")

In [0]:
medidas = medidas.withColumnRenamed("TAG_prefix", "ID_prefix")

In [0]:
df = arqlmed.join(
    medidas.select("ID_prefix", "H_LIM_C", "H_LIM_V"),
    on="ID_prefix",
    how="inner"
)

In [0]:
display(df)

In [0]:
# Match on first 6 chars of ID_prefix vs first 6 chars of TAGCOM
# This is the same approach used in older_files/Connecting weather to objects.ipynb

mds_slim = mds.select(
    F.substring("TAGCOM", 1, 6).alias("mds_key"),
    "CONCELHO",
    "X_SIT",
    "Y_SIT"
    ).dropDuplicates(["mds_key"])

df = df.withColumn("mds_key", F.substring("ID_prefix", 1, 6))

df = df.join(F.broadcast(mds_slim), on="mds_key", how="inner").drop("mds_key")

In [0]:
dbutils.data.summarize(df)

In [0]:
display(df)

In [0]:
# Snap DATE to the hour to match weather granularity (hourly)
df = df.withColumn("DATE_HOUR", F.date_trunc("hour", F.col("DATE")))

weather_slim = weather.select(
    F.col("DATE").alias("DATE_HOUR"),
    "location",
    "temperatura_media_do_ar_horaria_c",
    "humidade_relativa_media_horaria_percent",
    "precipitacao_horaria_mm",
    "velocidade_do_vento_media_horaria_m/s"
)

In [0]:
# Map each ID to its nearest weather station via X_SIT, Y_SIT
# Reuse haversine approach: get distinct ID locations and crossjoin with weather stations

id_locations = (df
    .select("ID_prefix", "X_SIT", "Y_SIT")
    .dropDuplicates(["ID_prefix"])
    .filter(F.col("X_SIT").isNotNull() & F.col("Y_SIT").isNotNull())
)

weather_stations = (weather
    .select("location", "latitude", "longitude")
    .dropDuplicates(["location"])
)

def haversine_km(lat1, lon1, lat2, lon2):
    r = F.lit(6371.0)
    dphi = F.radians(lat2 - lat1)
    dlambda = F.radians(lon2 - lon1)
    a = (
        F.pow(F.sin(dphi / 2), 2)
        + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlambda / 2), 2)
    )
    return r * 2 * F.asin(F.sqrt(a))

w_dist = Window.partitionBy("ID_prefix").orderBy("dist_km")

id_to_station = (
    id_locations.crossJoin(F.broadcast(weather_stations))
    .withColumn("dist_km", haversine_km(
        F.col("Y_SIT"), F.col("X_SIT"),       # lat=Y_SIT, lon=X_SIT per MDS convention
        F.col("latitude"), F.col("longitude")
    ))
    .withColumn("rn", F.row_number().over(w_dist))
    .filter(F.col("rn") == 1)
    .select("ID_prefix", "location")
)

df = df.join(F.broadcast(id_to_station), on="ID_prefix", how="inner")

df = df.join(
    weather_slim,
    on=["DATE_HOUR", "location"],
    how="inner"
).drop("DATE_HOUR", "location")

In [0]:
display(df)

In [0]:
df = df.select(
    "ID_prefix",
    "DATE",
    "current",
    "voltage",
    "H_LIM_C",
    "H_LIM_V",
    "CONCELHO",
    "X_SIT",
    "Y_SIT",
    "temperatura_media_do_ar_horaria_c",
    "humidade_relativa_media_horaria_percent",
    "precipitacao_horaria_mm",
    "velocidade_do_vento_media_horaria_m/s"
)

In [0]:
# silver_event_log already has: ID_prefix, DATE_15M, events_15m_cnt

df = df.join(
    event_log.select("ID_prefix", "DATE", "events_15m_cnt"),
    on=["ID_prefix", "DATE"],
    how="left"
)

# Fill nulls with 0 — no events in that bucket means 0
df = df.fillna(0, subset=["events_15m_cnt"])

In [0]:
display(df)

In [0]:
dbutils.data.summarize(df)

In [0]:
target_catalog = "hive_metastore"
target_schema  = "gold"
target_table   = "gold_dataset"

full_name = f"{target_catalog}.{target_schema}.{target_table}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

(df.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(full_name))

print(f"✅ Saved {full_name} — {df.count():,} rows")